# Generate your requirements file

This notebook exports the current environment's package requirements to a `requirements` file. This file is really important as you will need it to upload it together with the notebook into your repository. Please, run the code cell bellow and follow the steps.

> **IMPORANT**: Make sure that you are running this notebook in the same environment you used to run the notebook you are uploading.

# Step 0: Import main functionalities

In [ ]:
# @markdown Run this cell to import main functionalities

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

# For the requirements extraction we will need nbformat and pyyaml installed, also check for them
try:
    import nbformat
except ImportError:
    raise ImportError("nbformat is not installed. Please install it using 'pip install nbformat' or 'conda install -c conda-forge nbformat'.")

try:
    import yaml
except ImportError:
    raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

from IPython.display import display
from pathlib import Path
import importlib.metadata
import subprocess
import contextlib
import importlib
import platform
import nbformat
import yaml
import re
import sys
import os
import io
import html
import shutil
import tempfile
import uuid

import_regex = r'^[ \t]*import .*'
from_regex = r'^[ \t]*from .* import .*'

LOCAL_SCAN_EXTENSIONS = {'.py'}
LOCAL_SCAN_LIMIT = 200  # fail-safe limit to avoid scanning huge trees
LOCAL_SCAN_PREVIEW = 8
GITHUB_CLONE_ROOT = Path(tempfile.gettempdir()) / "requirements_helper_repos"


def extract_code_import_lines(code):
    list_import_lines = []

    # We are going line by line analyzing them
    lines = code.split('\n')
    for line in lines:
        if re.match(import_regex, line) or re.match(from_regex, line):
            list_import_lines.append(line.strip())
    return list_import_lines

def extract_imported_packages(code):
    list_import_packages = []
    lines = code.split('\n')
    for line in lines:
        line = line.strip()

        # Handle "import X" or "import X, Y" or "import X as Y"
        if re.match(import_regex, line):
            # Extract package name after 'import'
            match = re.search(r'import\s+(.+)', line)
            if match:
                packages = match.group(1).split(',')  # Handle multiple imports
                for pkg in packages:
                    pkg = pkg.split(' as ')[0].strip()  # Remove "as alias"
                    pkg = pkg.split('.')[0]  # Get root package only
                    if pkg:
                        list_import_packages.append(pkg)

        # Handle "from X import Y"
        elif re.match(from_regex, line):
            # Extract module name after 'from'
            match = re.search(r'from\s+([^\s]+)\s+import', line)
            if match:
                module = match.group(1).split('.')[0]  # Get root package
                if module and module != '.':  # Skip relative imports
                    list_import_packages.append(module)

    return list_import_packages

def extract_pip_install(code):
    list_pip_install_lines = []
    lines = code.split('\n')

    for line in lines:
        line = line.strip()

        # Skip empty lines and comments
        if not line or line.startswith('#'):
            continue

        # Check if line starts with ! or %
        if line.startswith('!') or line.startswith('%'):
            # Remove the leading ! or %
            clean_command = line[1:].strip()

            # Verify it contains 'install' keyword
            if 'install' in clean_command:
                # Remove trailing comments if present
                if '#' in clean_command:
                    clean_command = clean_command.split('#')[0].strip()

                list_pip_install_lines.append(clean_command)

    return list_pip_install_lines

def is_builtin_or_stdlib(package_name):
    """Check if a package is a built-in or standard library module."""
    if package_name in sys.builtin_module_names:
        return True

    if hasattr(sys, 'stdlib_module_names') and package_name in sys.stdlib_module_names:
        return True

    # Fallback: common stdlib modules for older Python versions
    stdlib_modules = {
        're', 'sys', 'os', 'platform', 'pathlib', 'json', 'math', 'random',
        'datetime', 'time', 'collections', 'itertools', 'functools', 'operator',
        'pickle', 'csv', 'sqlite3', 'threading', 'subprocess', 'shutil', 'glob',
        'io', 'codecs', 'logging', 'argparse', 'configparser', 'tempfile',
        'urllib', 'http', 'email', 'base64', 'hashlib', 'hmac', 'secrets',
        'struct', 'textwrap', 'string', 'warnings', 'contextlib', 'abc',
        'importlib', 'inspect', 'traceback', 'unittest', 'doctest', 'pdb',
        'profile', 'pstats', 'timeit', 'statistics', 'asyncio', 'concurrent',
        'socket', 'ssl', 'select', 'selectors', 'queue', 'multiprocessing',
        'copy', 'types', 'enum', 'dataclasses', 'typing', 'weakref',
        'array', 'heapq', 'bisect', 'graphlib', 'decimal', 'fractions',
        'numbers', 'cmath', 'zlib', 'gzip', 'bz2', 'lzma', 'tar', 'zipfile',
        'netrc', 'xdrlib', 'plistlib', 'html', 'xml',
        'webbrowser', 'cgi', 'wsgiref', 'uuid', 'socketserver', 'xmlrpc',
    }

    return package_name in stdlib_modules


def is_environment_specific(package_name):
    """Check if a package is environment-specific and shouldn't be in requirements."""
    environment_specific = {
        'google',
        'google.colab',
        'colab',
        'IPython',
        'ipykernel',
        'jupyter',
        'jupyterlab',
    }
    return package_name in environment_specific


def package_exists_on_pypi(package_name):
    """Check if a package exists on PyPI using the PyPI JSON API."""
    try:
        import requests
        response = requests.get(f"https://pypi.org/pypi/{package_name}/json", timeout=2)
        return response.status_code == 200
    except:
        return True


def get_package_version(package_name):
    try:
        module = importlib.import_module(package_name)
        return module.__version__
    except AttributeError:
        try:
            return importlib.metadata.version(package_name)
        except:
            return ""
    except:
        return ""


def get_package_name_for_pip(import_name):
    """
    Convert import name to pip package name.
    E.g., 'docx' -> 'python-docx', 'cv2' -> 'opencv-python'
    """
    known_mappings = {
        'docx': 'python-docx',
        'cv2': 'opencv-python',
        'skimage': 'scikit-image',
        'PIL': 'Pillow',
        'yaml': 'pyyaml',
        'sklearn': 'scikit-learn',
        'bs4': 'beautifulsoup4',
        'dotenv': 'python-dotenv',
        'fpdf': 'fpdf2',
    }

    if import_name in known_mappings:
        return known_mappings[import_name]

    # Try to find via installed distributions
    try:
        for dist in importlib.metadata.distributions():
            try:
                top_level = dist.read_text('top_level.txt')
                if top_level and import_name in top_level.split('\n'):
                    return dist.name
            except:
                pass
    except:
        pass

    # Fallback: assume import name is correct
    return import_name


def load_pip_freeze_requirements():
    mapping = {}
    try:
        result = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    except Exception:
        return mapping

    for raw_line in result.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue

        requirement = line
        key = None

        if line.startswith('-e '):
            entry = line[3:].strip()
            if '#egg=' in entry:
                key = entry.split('#egg=', 1)[1]
            elif ' @ ' in entry:
                key = entry.split(' @ ', 1)[0]
            requirement = entry

        elif ' @ ' in line:
            key = line.split(' @ ', 1)[0]

        elif '==' in line:
            key = line.split('==', 1)[0]

        elif '#egg=' in line:
            key = line.split('#egg=', 1)[1]

        else:
            key = line

        if key:
            mapping[key.strip().lower()] = requirement

    return mapping


def requirement_from_pip_freeze(import_name, pip_freeze_requirements):
    lower_name = import_name.lower()
    if lower_name in pip_freeze_requirements:
        return pip_freeze_requirements[lower_name]

    pip_name = get_package_name_for_pip(import_name)
    if pip_name and pip_name.lower() in pip_freeze_requirements:
        return pip_freeze_requirements[pip_name.lower()]

    return None

def sanitize_path_input(raw):
    """
    Turn a user-pasted path into a safe Path without using `os`.
    - Strips ASCII + smart quotes and surrounding punctuation
    - Expands ~ to the user home directory
    - Replaces common Windows placeholders (%USERPROFILE%, %HOMEPATH%) with the home dir
    - Normalises mixed slashes
    """
    s = str(raw or "").strip()
    if not s:
        return Path("")

    # Remove all kinds of quotes people paste (ASCII and curly/smart quotes)
    s = re.sub(r'[\"\'\u201c\u201d\u2018\u2019]', "", s)

    # Trim accidental leading/trailing punctuation or commas
    s = s.strip(" \t\n\r,;<>")

    # Expand ~ at the start to the user's home dir
    home = Path.home().as_posix()
    if s == "~":
        s = home
    elif s.startswith("~/") or s.startswith("~\\"):
        s = home + s[1:]

    # Common Windows env placeholders -> expand to home (no os.environ access here)
    # This covers the most common pasted forms like %USERPROFILE%\Documents
    s = s.replace("%USERPROFILE%", home)
    s = s.replace("%HOMEPATH%", home)

    # Replace repeated backslash-quote artefacts that sometimes appear from copy/paste
    s = s.replace('\\"', '').replace("\\'", "")

    # Normalise forward/back slashes to the OS-agnostic form; Path() will handle exact OS semantics
    s = s.replace("/", str(Path.home()).startswith("\\") and "\\" or "/") if False else s  # noop kept explicit: Path handles slashes

    # Final strip (in case replacements introduced new edge whitespace/punctuation)
    s = s.strip(" \t\n\r,;<>")

    return Path(s)


def parse_local_path_entries(multiline_text):
    """
    Convert newline-separated user input into Path objects that exist on disk.
    Returns a tuple (paths, errors).
    """
    if not multiline_text:
        return [], []

    parsed_paths = []
    errors = []
    seen = set()

    for raw_line in multiline_text.splitlines():
        trimmed = raw_line.strip()
        if not trimmed:
            continue

        candidate = sanitize_path_input(trimmed).expanduser()
        if not str(candidate):
            continue

        if not candidate.exists():
            errors.append(f"{trimmed} (path not found)")
            continue

        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate

        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        parsed_paths.append(resolved)

    return parsed_paths, errors


def expand_local_entries(entries, limit=LOCAL_SCAN_LIMIT):
    """
    Expand a list of file/folder entries into individual files we can scan.
    Returns (files, warnings). Only .py files are considered.
    """
    expanded_files = []
    warnings = []
    seen = set()

    def register_file(file_path):
        try:
            resolved = file_path.resolve()
        except Exception:
            resolved = file_path

        key = str(resolved)
        if key in seen:
            return True

        if limit and len(expanded_files) >= limit:
            return False

        seen.add(key)
        expanded_files.append(resolved)
        return True

    for entry in entries:
        try:
            if entry.is_file():
                if entry.suffix.lower() in LOCAL_SCAN_EXTENSIONS:
                    if not register_file(entry):
                        warnings.append(
                            f"Reached the scan limit ({limit} files). Remaining files were skipped."
                        )
                        break
                else:
                    warnings.append(f"{entry} (unsupported file type)")
            elif entry.is_dir():
                found_any = False
                for suffix in LOCAL_SCAN_EXTENSIONS:
                    for candidate in sorted(entry.rglob(f"*{suffix}")):
                        found_any = True
                        if not register_file(candidate):
                            warnings.append(
                                f"Reached the scan limit ({limit} files). Remaining files were skipped."
                            )
                            return expanded_files, warnings
                if not found_any:
                    warnings.append(f"{entry} (no .py files found)")
            else:
                warnings.append(f"{entry} (not a file or folder)")
        except Exception as exc:
            warnings.append(f"{entry}: {exc}")

    return expanded_files, warnings



def _create_clone_destination():
    GITHUB_CLONE_ROOT.mkdir(parents=True, exist_ok=True)
    return GITHUB_CLONE_ROOT / f"repo_{uuid.uuid4().hex}"

def clone_github_repository(repo_url, branch=None):
    if shutil.which('git') is None:
        raise RuntimeError("Git is not available in this environment. Please install Git to add helper repositories.")
    destination = _create_clone_destination()
    cmd = ['git', 'clone', '--depth', '1']
    if branch:
        cmd += ['--branch', branch]
    cmd += [repo_url, str(destination)]
    try:
        subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Git clone failed: " + exc.output.strip()) from exc
    except FileNotFoundError:
        raise RuntimeError("Git is not installed or not on PATH. Please install Git to add helper repositories.")
    return destination

def fetch_github_helper_path(repo_url, repo_subpath="", branch=""):
    repo_url = (repo_url or '').strip()
    if not repo_url:
        raise ValueError("Please provide a GitHub repository URL.")
    repo_subpath = (repo_subpath or '').strip().strip('\\/')
    branch = (branch or '').strip()

    cloned_repo = clone_github_repository(repo_url, branch or None)
    helper_root = cloned_repo / repo_subpath if repo_subpath else cloned_repo

    if not helper_root.exists():
        raise FileNotFoundError(
            f"Path '{repo_subpath or '.'}' was not found inside the repository {repo_url}."
        )

    return helper_root

def resolve_ipywidgets_version(py_version):
    try:
        py_major, py_minor = map(int, py_version.split(".")[:2])
        
        if py_major >= 3 and py_minor >= 9:
            return "8.1.7"  # Latest stable for modern Python + JupyterLab 4
        elif py_major >= 3 and py_minor >= 7:
            return "8.1.6"  # Good for Python 3.7+
        else:
            return "7.7.2"  # Fallback for older environments
    except Exception:
        return ">=7.0.0"  # Safe fallback

def is_colab():
    return "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ


def render_section(content_container, *objs):
    """Render downstream UI inside a single widget container (or fallback to display)."""
    if content_container is not None:
        content_container.children = tuple(objs)
    else:
        display(*objs)

display(widgets.HTML("<b>✅ Core functionalities loaded! Start with Step 1.</b>"))

# Step 1: Select the notebook you want to extract requirements from

In [ ]:
# @markdown Run this cell to choose notebook

colab_flag = is_colab()

mount_title = widgets.HTML(
    "<h2>1.- Click the button to mount your Google Drive</h2>"
)
mount_btn = widgets.Button(
    description="Mount Google Drive",
    button_style="info",
    layout=widgets.Layout(width="100%")
)
mount_status = widgets.HTML("")

colab_find_notebook = widgets.HTML(
    "<h2>2.- Find your notebook</h2>"
    "<ol>"
    "<li>On the left sidebar, click on the folder icon to open the file explorer.</li>"
    "<li>You should see a `drive` folder, otherwise go back to Step 1.</li>"
    "<li>Go to <b>drive</b> > <b>MyDrive</b>. There you will find your Google Drive files.</li>"
    "<li>Locate the notebook you want to extract requirements from. Notebook are stored by default in <b>Colab Notebooks</b> folder.</li>"
    "</ol>"

    "<h4>What if I cannot find my notebook?</h4>"
    "It might happen that your notebook is not stored in Google Drive. <b>Do you have it stored locally on your computer?</b>"
    "<ul>"
    "<li>If <b>yes</b>, drag and drop it in Colab's file explorer for a single use or upload it to Google Drive for persistent storage.</li>"
    "<li>If <b>not</b>, you can download it from wherever you have it stored (e.g., GitHub, email, etc.) and then upload it to Google Drive or directly to Colab.</li>"
    "</ul>"
    "In case the notebook is in another Colab session you can always download it following these steps:"
    "<ol>"
    "<li>Open the Colab session where your notebook is.</li>"
    "<li>On top, go to <b>File</b> > <b>Download</b> > <b>Download .ipynb</b>.</li>"
    "</ol>"
)
colab_path_title = widgets.HTML(
    "<h2>3.- Provide a notebook path</h2>"
    "Right-click on the notebook on Colab's file explorer and select 'Copy path'.<br>"
    "Then paste the full path below and click 'Use this notebook'."
)

local_find_notebook = widgets.HTML(
    "<h2>1.- Locate your local notebook</h2>"
    "Please make sure you know the full path to your local notebook file (with .ipynb extension)."
)

local_path_title = widgets.HTML(
    "<h2>2.- Provide a notebook path</h2>"
    "Please paste the full path to your local notebook below and click 'Use this notebook'."
)
path_widget = widgets.Text(
    placeholder="/content/drive/MyDrive/your_notebook.ipynb" if colab_flag else "path/to/your_notebook.ipynb",
    description="Notebook path:",
    layout=widgets.Layout(width="99%"),
    style={'description_width': 'initial'}
)
path_status = widgets.HTML("")

use_btn = widgets.Button(
    description="Use this notebook",
    button_style="success",
    layout=widgets.Layout(width="99%")
)

nb_path = None

def on_mount(_):
    try:
        from google.colab import drive
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            drive.mount("/content/drive", force_remount=True)
        mount_status.value = "<b>✅ Drive mounted.</b> Update the path if needed, then click 'Use this notebook'."
    except Exception as e:
        mount_status.value = f"<b>❌ Drive mount failed:</b> {str(e)}"

def on_use(_):
    global nb_path
    nb_path = sanitize_path_input(path_widget.value).expanduser()

    # Allow folder input: if it's a directory, try to find .ipynb inside it
    if nb_path.exists() and nb_path.is_dir():
        nbs = sorted(nb_path.glob("*.ipynb"))
        if len(nbs) == 1:
            nb_path = nbs[0]
        elif len(nbs) > 1:
            path_status.value = "<b>❌ Multiple notebooks found in that folder. Please paste the full .ipynb path.</b>"
            return
        else:
            path_status.value = "<b>❌ No .ipynb files found in that folder. Please paste a full .ipynb path.</b>"
            return

    if not nb_path.exists() or nb_path.suffix != ".ipynb":
        if colab_flag:
            path_status.value = (
                "<b>❌ Invalid path.</b> Please paste a valid .ipynb path and try again, e.g., "
                "<code>/content/drive/MyDrive/Colab Notebooks/your_notebook.ipynb</code>"
            )
        else:
            path_status.value = (
                "<b>❌ Invalid path.</b> Please paste an absolute path to a valid local .ipynb path and try again. For example: "
                "<ul>"
                r"<li><code>C:\Users\YourName\Notebooks\your_notebook.ipynb</code> (Windows)</li>"
                "<li><code>/home/yourname/notebooks/your_notebook.ipynb</code> (Linux/Mac)</li>"
                "</ul>"
            )
        return

    path_status.value = f"<b>✅ You have chosen:</b> <code>{nb_path}</code>"
    path_status.value += "<br><b>If the notebook is correct, please go to Step 2.</b>"

if 'import_regex' not in globals():
    display(widgets.HTML("<b>❌ Main functionalities not loaded. Please go to Step 0 to import them.</b>"))
else:
    if colab_flag:
        mount_btn.on_click(on_mount)
    use_btn.on_click(on_use)

    if colab_flag:
        layout_children = [
            mount_title,
            mount_btn,
            mount_status,
            colab_find_notebook,
            colab_path_title,
        ]
    else:
        layout_children = [
            local_find_notebook,
            local_path_title,
        ]

    layout_children.extend([path_widget, use_btn, path_status])
    container = widgets.VBox(layout_children, layout=widgets.Layout(width="100%"))
    display(container)

# Step 2: Choose whether to include additional helper files (local or GitHub) for the requirements extraction


In [ ]:
# @markdown Run this cell to choose whether to include additional local files for the requirements extraction
if 'import_regex' not in globals():
    display(widgets.HTML("<b>&#x274C; Main functionalities not loaded. Please go to Step 0 to import them.</b>"))
elif 'nb_path' not in globals() or nb_path is None:
    display(widgets.HTML("<b>&#x274C; No notebook selected. Please go to Step 1 and select a notebook to extract requirements.</b>"))
else:
    instructions = widgets.HTML(
        f"<h3>💻 Include helper modules stored locally</h3>"
        "<p>If your notebook imports functions or classes from local <code>.py</code> files, list those paths below (one per line) so we can scan them for dependencies.</p>"
        "<ul>"
        "<li>Paths can be absolute or relative to the current working directory.</li>"
        "<li>Only files ending in <code>.py</code> are processed. Notebook files and other extensions are ignored.</li>"
        "<li>Folders are scanned recursively for <code>.py</code> files.</li>"
        f"<li>To keep things fast we scan up to {LOCAL_SCAN_LIMIT} files.</li>"
        "<li>Leave the field empty or click skip if there are no helper modules.</li>"
        "</ul>"
    )
    paths_input = widgets.Textarea(
        placeholder="e.g.\nsrc/helpers.py\ncustom_layers/\n../shared/utils/data_loader.py",
        description='Local files/folders:',
        layout=widgets.Layout(width='99%', height='140px'),
        style={'description_width': 'initial'}
    )
    scan_btn = widgets.Button(
        description='Scan paths and continue',
        button_style='info',
        layout=widgets.Layout(width='50%')
    )
    skip_btn = widgets.Button(
        description='Skip (no extra files)',
        layout=widgets.Layout(width='50%')
    )
    status = widgets.HTML()

    github_instructions = widgets.HTML(
        "<h3>🌐 Include helper modules from GitHub repositories</h3>"
        "<p>If the helper <code>.py</code> files live in another repository, provide its URL along with the folder (or file) path inside it.</p>"
        "<ul>"
        "<li>Repositories are downloaded to a temporary folder for this session.</li>"
        "<li>Only <code>.py</code> files in the paths you specify are scanned.</li>"
        "<li>Optionally set a branch name if you do not want the default branch.</li>"
        "<li>You can add as many repositories as needed.</li>"
        "</ul>"
    )
    repo_url_input = widgets.Text(
        placeholder="https://github.com/owner/repo.git",
        description='Repository URL:',
        layout=widgets.Layout(width='99%'),
        style={'description_width': 'initial'}
    )
    repo_branch_input = widgets.Text(
        placeholder="main (optional)",
        description='Branch (optional):',
        layout=widgets.Layout(width='99%'),
        style={'description_width': 'initial'}
    )
    repo_subpath_input = widgets.Text(
        placeholder="path/in/repo (leave blank for entire repo)",
        description='Helper path:',
        layout=widgets.Layout(width='99%'),
        style={'description_width': 'initial'}
    )
    add_repo_btn = widgets.Button(
        description='Add GitHub helper files',
        button_style='info',
        layout=widgets.Layout(width='50%')
    )
    clear_repo_btn = widgets.Button(
        description='Clear GitHub helpers',
        layout=widgets.Layout(width='50%')
    )
    github_status = widgets.HTML()
    github_summary = widgets.HTML("<i>No GitHub helper folders added yet.</i>")

    entries = []
    local_entries = []
    github_entries = []
    github_records = []

    def update_entries():
        global entries
        combined = []
        seen = set()
        for collection in (local_entries, github_entries):
            for path in collection:
                key = str(path)
                if key in seen:
                    continue
                seen.add(key)
                combined.append(path)
        entries = combined

    def render_github_summary():
        if not github_records:
            return "<i>No GitHub helper folders added yet.</i>"
        value = "<ul>"
        for record in github_records:
            repo = html.escape(record['repo'])
            branch = html.escape(record['branch']) if record['branch'] else ""
            subpath = html.escape(record['subpath'])
            local_path = html.escape(record['local'])
            branch_suffix = f" ➡️ <b>branch: {branch}</b>" if branch else ""
            value += f"<li><b>{repo}</b>{branch_suffix} ➡️ <b>{subpath}</b></li>"
        value += "</ul>"
        return value

    update_entries()

    def handle_scan(_):
        parsed, errors = parse_local_path_entries(paths_input.value)
        local_entries.clear()
        local_entries.extend(parsed)
        update_entries()
        if errors:
            message = "<b>❌ We could not find these paths:</b><ul>"
            for err in errors:
                message += f"<li>{html.escape(err)}</li>"
            message += "</ul>"
            status.value = message
            return
        if local_entries:
            status.value = "<b>✅ Your additional modules were successfully included and will be processed!</b>"
            status.value += "<br><b>✅ Now, check if you want to include more helper files and if not, please go to the Step 3 to continue.</b>"
        else:
            status.value = "<b>⚠️ No valid paths were provided. Only the notebook will be analyzed for requirements.</b>"
            status.value += "<br><b>✅ Now, check if you want to include more helper files and if not, please go to the Step 3 to continue.</b>"

    def handle_skip(_):
        local_entries.clear()
        update_entries()
        status.value = "<b>⚠️ The scan was skipped. Only the notebook and any GitHub helpers you add will be analyzed.</b>"
        status.value += "<br><b>✅ Now, check if you want to include more helper files and if not, please go to the Step 3 to continue.</b>"

    def handle_add_repo(_):
        repo_url = repo_url_input.value.strip()
        branch = repo_branch_input.value.strip()
        subpath = repo_subpath_input.value.strip()
        if not repo_url:
            github_status.value = "<b>❌ Please provide a valid GitHub repository URL.</b>"
            return
        try:
            helper_path = fetch_github_helper_path(repo_url, subpath, branch)
        except Exception as exc:
            github_status.value = "<b>❌ Unable to add GitHub helper files:</b> " + html.escape(str(exc))
            return
        helper_key = str(helper_path)
        if any(str(path) == helper_key for path in github_entries):
            github_status.value = "<b>⚠️ That repository path has already been added.</b>"
            return
        github_entries.append(helper_path)
        github_records.append({
            'repo': repo_url,
            'branch': branch,
            'subpath': subpath or '.',
            'local': helper_key,
        })
        update_entries()
        github_summary.value = render_github_summary()
        github_status.value = "<b>✅ GitHub helper files added. They will be processed with your notebook.</b>"
        github_status.value += "<br><b>✅ Now, check if you want to include more helper files and if not, please go to the Step 3 to continue.</b>"

    def handle_clear_repos(_):
        if not github_entries:
            github_status.value = "<b>⚠️ There are no GitHub helper entries to clear.</b>"
            return
        github_entries.clear()
        github_records.clear()
        update_entries()
        github_summary.value = render_github_summary()
        github_status.value = "<b>⚠️ GitHub helper selections cleared. Only the notebook and local paths will be analyzed.</b>"

    scan_btn.on_click(handle_scan)
    skip_btn.on_click(handle_skip)
    add_repo_btn.on_click(handle_add_repo)
    clear_repo_btn.on_click(handle_clear_repos)

    buttons_box = widgets.HBox(
        [scan_btn, skip_btn],
        layout=widgets.Layout(width='99%', justify_content='space-between')
    )

    repo_buttons_box = widgets.HBox(
        [add_repo_btn, clear_repo_btn],
        layout=widgets.Layout(width='99%', justify_content='space-between')
    )

    repo_inputs_box = widgets.VBox(
        [repo_url_input, repo_branch_input, repo_subpath_input],
        layout=widgets.Layout(width='99%')
    )

    github_summary.value = render_github_summary()

    display(
        widgets.HTML("<h2>📥 Include helper modules:</h2>"),
        github_instructions,
        repo_inputs_box,
        repo_buttons_box,
        github_status,
        github_summary,
        instructions,
        paths_input,
        buttons_box,
        status,
    )


# Step 3: Check if provided files contain any Pypi installations

In [ ]:
# @markdown Run this cell to check if provided files contain any Pypi installations
def check_pip_install(notebook_path, additional_entries):
    try:
        colab_nb = nbformat.read(notebook_path, as_version=4)
    except Exception as exc:
        raise RuntimeError("Unable to read the selected notebook: " + str(exc))

    try:
        local_dependency_files, local_scan_warnings = expand_local_entries(
            additional_entries,
            limit=LOCAL_SCAN_LIMIT,
        )
    except Exception as exc:
        raise RuntimeError("Failed to scan the provided paths: " + str(exc))

    pip_install_lines = []

    for cell in colab_nb.cells:
        if cell.cell_type == "code":
            pip_install_lines.extend(extract_pip_install(cell.source))

    for local_file in local_dependency_files:
        suffix = local_file.suffix.lower()
        try:
            if suffix == ".py":
                try:
                    code_blob = local_file.read_text(encoding="utf-8")
                except UnicodeDecodeError:
                    code_blob = local_file.read_text(encoding="utf-8", errors="ignore")
                pip_install_lines.extend(extract_pip_install(code_blob))
            else:
                continue
        except Exception as exc:
            local_scan_warnings.append(f"{local_file}: {exc}")

    # Remove duplicates while preserving order
    pip_install_lines = list(dict.fromkeys(pip_install_lines))

    display(widgets.HTML(f"<h2>Detected pip install commands</h2>"))

    if pip_install_lines:
        from IPython import get_ipython
        get_ipython().set_next_input("print('hello')", replace=False)
        explanation_html = widgets.HTML(
            "<h3>The following pip install commands were detected in the notebook or the scanned helper <code>.py</code> files.</h3>"
            "Please ensure these packages are installed in order to be detected and included in the generated requirements.yaml file.<br>"
            "If you want to install them now, we have facilitated you a code cell bellow this one with the code to install them, so please:"
            "<ol>"
            "<li>Run the code cell below.</li>"
            "<li>Wait for the installation to complete.</li>"
            "<li>Delete the code cell and go to Step 4.</li>"
            "</ol>"
            "If you have already installed them, please delete the code cell bellow with the installation commands and <b>continue to Step 4.</b>"
        )
        pip_installed_textarea = widgets.Textarea(
            value="\n".join([f"!{line}" for line in pip_install_lines]),
            description='Detected pip installs:',
            layout=widgets.Layout(width='99%', height='100px'),
            style={'description_width': 'initial'}
        )

        installing_text = "\n".join([f"!{line}" for line in pip_install_lines])
        installing_text += "\nprint('✅ Installation complete! Please, delete this code cell and go to Step 4.')"
        get_ipython().set_next_input(installing_text, replace=False)

        display(explanation_html, pip_installed_textarea)
    else:
        display(widgets.HTML("<b>No pip install commands detected in the notebook or the scanned helper <code>.py</code> files.</b><br><b>✅ Please continue to Step 4.</b>"))

if 'import_regex' not in globals():
    display(widgets.HTML("<b>❌ Main functionalities not loaded. Please go to Step 0 to import them.</b>"))
elif 'nb_path' not in globals() or nb_path is None:
    display(widgets.HTML("<b>❌ No notebook selected. Please go to Step 1 and select a notebook to extract requirements.</b>"))
elif 'entries' not in globals() or entries is None:
    display(widgets.HTML("<b>❌ No information about additional helper files (local or GitHub). Please go to Step 2 and choose whether to include additional helper files for the requirements extraction.</b>"))
else:
    check_pip_install(nb_path, entries)

# Step 4: Extract dependencies and generate the requirements file

In [ ]:
# @markdown Run this cell to extract depenencies and generate the requirements file
def process_notebook(additional_entries, content_container, notebook_path, pip_freeze_requirements):
    loading = widgets.HTML("<b>Scanning notebook and selected files...</b>")
    render_section(content_container, loading)

    try:
        local_dependency_files, local_scan_warnings = expand_local_entries(
            additional_entries,
            limit=LOCAL_SCAN_LIMIT,
        )
    except Exception as exc:
        render_section(
            content_container,
            widgets.HTML(
                "<b>❌ Failed to scan the provided paths:</b> "
                + html.escape(str(exc))
            )
        )
        return

    try:
        colab_nb = nbformat.read(notebook_path, as_version=4)
    except Exception as exc:
        render_section(
            content_container,
            widgets.HTML(
                "<b>❌ Unable to read the selected notebook:</b> "
                + html.escape(str(exc))
            )
        )
        return

    # Gather imports
    all_imports = []

    for cell in colab_nb.cells:
        if cell.cell_type == "code":
            all_imports.extend(extract_imported_packages(cell.source))
    for local_file in local_dependency_files:
        suffix = local_file.suffix.lower()
        try:
            if suffix == ".py":
                try:
                    code_blob = local_file.read_text(encoding="utf-8")
                except UnicodeDecodeError:
                    code_blob = local_file.read_text(encoding="utf-8", errors="ignore")
                all_imports.extend(extract_imported_packages(code_blob))
            else:
                continue
        except Exception as exc:
            local_scan_warnings.append(f"{local_file}: {exc}")

    # Remove duplicates while preserving order
    all_imports = list(dict.fromkeys(all_imports))

    # Process packages to get versions
    versioned_packages = []
    not_found_packages = []
    filtered_packages = []

    for pkg in all_imports:
        normalized_pkg = pkg.strip()

        if is_builtin_or_stdlib(normalized_pkg):
            continue

        if is_environment_specific(normalized_pkg):
            filtered_packages.append(f"{normalized_pkg} (environment-specific)")
            continue

        freeze_requirement = requirement_from_pip_freeze(
            normalized_pkg, pip_freeze_requirements
        )
        pip_name = get_package_name_for_pip(normalized_pkg)

        if freeze_requirement:
            versioned_packages.append(freeze_requirement)
            continue

        try:
            version = get_package_version(normalized_pkg)
        except Exception:
            version = ""

        if not version and pip_name and pip_name != normalized_pkg:
            version = get_package_version(pip_name)

        if pip_name and not package_exists_on_pypi(pip_name):
            filtered_packages.append(f"{pip_name} (not on PyPI)")
            continue

        resolved_name = pip_name or normalized_pkg

        if version:
            versioned_packages.append(f"{resolved_name}=={version}")
        else:
            not_found_packages.append(resolved_name)

    # Get the current Python version
    python_version = platform.python_version()

    def build_local_files_html():
        if not local_dependency_files:
            return ""
        preview_paths = local_dependency_files[:LOCAL_SCAN_PREVIEW]
        value = "<b>Local files scanned:</b><br><ul>"
        for path in preview_paths:
            value += f"<li>{html.escape(str(path))}</li>"
        remaining = len(local_dependency_files) - len(preview_paths)
        if remaining > 0:
            value += f"<li>... and {remaining} more</li>"
        value += "</ul>"
        return value

    def build_local_warnings_html():
        if not local_scan_warnings:
            return ""
        value = "<b>Local scan notes:</b><br><ul>"
        for note in local_scan_warnings:
            value += f"<li>{html.escape(str(note))}</li>"
        value += "</ul>"
        return value

    local_files_html_value = build_local_files_html()
    local_warnings_html_value = build_local_warnings_html()

    def show_requirements_dialog():
        """Display the version info and generate requirements dialog."""
        version_info = widgets.HTML(
            "<h2>✅ Requirements correctly extracted!</h2>"
            "<p>These are the packages that were detected from the notebook and any optional helper <code>.py</code> files you provided.</p>"
        )

        if local_files_html_value:
            version_info.value += local_files_html_value

        version_info.value += "<h3>✅ Found packages with versions:</h3>"
        version_info.value += "<p>These packages were found on your environment with version information and will be included in the generated requirements file.</p>"
        version_info.value += "<ul>"
        for vp in versioned_packages:
            version_info.value += f"<li>{html.escape(vp)}</li>"
        version_info.value += "</ul>"

        if filtered_packages:
            version_info.value += "<h3>⚠️ Filtered packages (excluded from requirements):</h3>"
            version_info.value += "<p>You should not worry too much about these packages. These are packages that were detected but will not be included in the generated requirements file because they are either environment-specific (e.g., <code>IPython</code>, <code>jupyterlab</code>, etc.) or not available on PyPI.</p>"
            version_info.value += "<ul>"
            for fp in filtered_packages:
                version_info.value += f"<li>{html.escape(fp)}</li>"
            version_info.value += "</ul>"

        if not_found_packages:
            version_info.value += "<h3>❌ Packages not found on your environment or without version info:</h3>"
            version_info.value += "<p><b>Careful:</b> These packages were detected but could not be found in your environment or do not have version information available. <br> Please, be sure that you are in the correct environment where you run the notebook or that the packages are installed. <br> In case they are not installed remember to install them with <code>pip install &lt;package_name&gt;</code> or <code>conda install &lt;package_name&gt;</code>.</p>"
            version_info.value += "<ul>"
            for np in not_found_packages:
                version_info.value += f"<li>{html.escape(np)}</li>"
            version_info.value += "</ul>"

        if local_warnings_html_value:
            version_info.value += local_warnings_html_value

        version_info.value += f"<h2>Detected Python version:</h2> {html.escape(python_version)}<br>"

        # Ask for a notebook description + generate requirements.yaml
        version_info.value += "<h2>Provide a description for your notebook:</h2>"
        version_info.value += "<p>This description will be included in the generated requirements file and can be helpful for documentation purposes or to give context about the notebook's purpose and functionality.</p>"
        version_info.value += "<p>After writing the description, click <b>Validate & preview requirements file</b> to inspect the YAML before using <b>Generate & save requirements file</b> to write it.</p>"
        default_notebook_description = f"This notebook '{notebook_path.name}' is for ..."

        description = widgets.Textarea(
            value=default_notebook_description,
            placeholder='Type something',
            description='Description:',
            disabled=False,
            layout=widgets.Layout(width='99%', height='80px')
        )

        preview_button = widgets.Button(
            description='Validate & preview requirements file',
            disabled=False,
            button_style='info',
            tooltip='Build a preview of the requirements.yaml file',
            icon='search',
            layout=widgets.Layout(width='99%')
        )
        generate_button = widgets.Button(
            description='Generate & save requirements file',
            disabled=True,
            button_style='success',
            tooltip='Preview your requirements before generating the file',
            icon='check',
            layout=widgets.Layout(width='99%')
        )
        preview_output = widgets.HTML()
        final_output = widgets.HTML()

        last_preview = {
            'payload': None,
            'description': None,
            'yaml': None,
            'warning': "",
        }

        def build_requirements_payload(description_text):
            prepared_packages = list(versioned_packages)
            added_ipywidgets = False
            ipywidgets_warning = ""
            ipywidget_version = resolve_ipywidgets_version(python_version)
            ipywidget_requirement = f"ipywidgets=={ipywidget_version}"
            for idx_pkg, pkg in enumerate(prepared_packages):
                if pkg.startswith("ipywidgets"):
                    prepared_packages[idx_pkg] = ipywidget_requirement
                    added_ipywidgets = True
                    break

            if not added_ipywidgets:
                prepared_packages.append(ipywidget_requirement)
            else:
                ipywidgets_warning = f"⚠️ Warning! The ipywidgets version has been overridden to {ipywidget_version}.<br>"

            payload = {
                'description': description_text,
                'python_version': python_version,
                'dependencies': prepared_packages,
                'windows_dependencies': [],
                'linux_dependencies': [],
                'macos_dependencies': [],
            }
            return payload, ipywidgets_warning

        def on_preview_clicked(_):
            preview_output.value = ""
            final_output.value = ""
            desc_text = description.value.strip()
            if not desc_text:
                preview_output.value = "<b>❌ Please provide a valid description before previewing.</b>"
                generate_button.disabled = True
                generate_button.tooltip = 'Preview your requirements before generating the file'
                return
            try:
                payload, warning = build_requirements_payload(desc_text)
            except Exception as exc:
                preview_output.value = "<b>❌ Failed to build the requirements preview:</b> " + html.escape(str(exc))
                generate_button.disabled = True
                generate_button.tooltip = 'Preview your requirements before generating the file'
                return

            preview_yaml = yaml.dump(payload, default_flow_style=False, sort_keys=False)
            preview_html = ""
            if warning:
                preview_html += warning
            # Look if the python version is bellow 3.11
            if tuple([int(e) for e in python_version.split('.')]) < (3, 11):
                preview_html += "⚠️ Warning! The Python version is below 3.11. The play button feature may not be supported (check <a href='https://github.com/CellMigrationLab/LabConstrictor/blob/main/.tools/docs/code_hiding.md#how-does-the-play-button-work'  target=\"_blank\" style=\"color: #0066cc; text-decoration: underline;\">here</a> to know more about the play button).<br>"
            preview_html += f"<b>Preview of {nb_path.stem}_requirements.yaml (nothing has been saved yet):</b>"
            preview_html += "<pre style='background:#1111110d;padding:12px;border-radius:6px;max-height:320px;overflow:auto;'>"
            preview_html += html.escape(preview_yaml)
            preview_html += "</pre>"
            preview_output.value = preview_html

            last_preview['payload'] = payload
            last_preview['description'] = desc_text
            last_preview['yaml'] = preview_yaml
            last_preview['warning'] = warning
            generate_button.disabled = False
            generate_button.tooltip = 'Generate requirements.yaml using the preview above'

        def on_generate_clicked(_):
            final_output.value = ""
            if not last_preview['payload']:
                final_output.value = "<b>❌ Please preview your requirements file before generating.</b>"
                generate_button.disabled = True
                generate_button.tooltip = 'Preview your requirements before generating the file'
                return
            current_description = description.value.strip()
            if current_description != (last_preview['description'] or ''):
                final_output.value = "<b>❌ The description has changed. Please preview again before generating the file.</b>"
                generate_button.disabled = True
                generate_button.tooltip = 'Preview your requirements before generating the file'
                return

            requirements_path = Path(f'{nb_path.stem}_requirements.yaml')
            try:
                with open(requirements_path, 'w') as file:
                    yaml.dump(last_preview['payload'], file, default_flow_style=False)
            except Exception as exc:
                final_output.value = "<b>❌ Failed to save the requirements file:</b> " + html.escape(str(exc))
                return

            success_message = f"<b>✅ Congrats! Your {requirements_path.name} file has been generated successfully!</b><br>"
            if last_preview['warning']:
                success_message = last_preview['warning'] + success_message
            if is_colab():
                from google.colab import files
                files.download(requirements_path)
                success_message += f'Your {requirements_path.name} file will be automatically downloaded. Additionally, you could find it in <b>{requirements_path.resolve()}</b><br>'
            else:
                success_message += f'You can find your {requirements_path.name} file in <a href="{requirements_path.resolve()}" target="_blank" style="color: #0066cc; text-decoration: underline;">{requirements_path.resolve()}</a>.<br>'
            success_message += "Please validate its content, edit if necessary (missed packages or edge cases) and in case you edited validate it using the Requirements Validator notebook."
            final_output.value = success_message

        preview_button.on_click(on_preview_clicked)
        generate_button.on_click(on_generate_clicked)

        render_section(content_container, 
                       version_info, description, preview_button, preview_output, generate_button, final_output)

    show_requirements_dialog()

if 'import_regex' not in globals():
    display(widgets.HTML("<b>❌ Main functionalities not loaded. Please go to Step 0 to import them.</b>"))
elif 'nb_path' not in globals() or nb_path is None:
    display(widgets.HTML("<b>❌ No notebook selected. Please go to Step 1 and select a notebook to extract requirements.</b>"))
elif 'entries' not in globals() or entries is None:
    display(widgets.HTML("<b>❌ No information about additional helper files (local or GitHub). Please go to Step 2 and choose whether to include additional helper files for the requirements extraction.</b>"))
else:
    content_container = widgets.VBox(layout=widgets.Layout(width="100%"))

    # Load pip-freeze mapping (do it here so it reflects the runtime env at click-time)
    pip_freeze_requirements = load_pip_freeze_requirements()

    try:
        process_notebook(entries, content_container, nb_path, pip_freeze_requirements)
    except Exception as exc:
        content_container.children = (
            widgets.HTML(
                "<b>❌ Failed to extract requirements:</b> "
                + str(exc)
            ),
        )

    container = widgets.VBox([content_container], layout=widgets.Layout(width="100%"))
    display(container)